In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, callbacks

Train = '/kaggle/input/datasets/tombackert/brain-tumor-mri-data/brain-tumor-mri-dataset'

model_train = image_dataset_from_directory(
    Train,
    batch_size = 32,
    image_size = (224, 224),
    validation_split = 0.2,
    subset = 'training',
    label_mode = 'categorical',
    seed = 123
)

model_val = image_dataset_from_directory(
    Train,
    batch_size = 32,
    image_size = (224, 224),
    validation_split = 0.2,
    subset = 'validation',
    label_mode = 'categorical',
    seed = 123
)

CLASS_NAMES = model_train.class_names



efficient_model = EfficientNetB0(
    weights = 'imagenet',
    include_top = False,
    input_shape = (224, 224, 3)
)

efficient_model.trainable = False



strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    
    augmentation = models.Sequential([
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.2)
    ])

    main_model = models.Sequential([
        augmentation,
        efficient_model,
        
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(len(CLASS_NAMES), activation='softmax')
    ])
    
    main_model.compile(
        optimizer = 'adam',
        loss = 'categorical_crossentropy',
        metrics = ['accuracy']
    )



reduce_speed = callbacks.ReduceLROnPlateau(
    monitor = 'val_loss',
    factor = 0.2,
    patience = 2,
    min_lr = 1e-6
)

early_stop = callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience = 3,
    restore_best_weights = True
)

main_model.fit(
    model_train,
    validation_data = model_val,
    epochs = 20,
    callbacks = [reduce_speed, early_stop]
)

main_model.save('brain_tumor_modele.h5')
main_model.summary()